In [ ]:
%%capture
!pip uninstall -y torchao bitsandbytes
!pip install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.44.2 peft accelerate datasets trl==0.9.4 PyGithub


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Using the base model since we dropped unsloth
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load in standard 16-bit float (Supported perfectly by P100 GPU)
# We drop 4-bit quantization entirely to avoid bitsandbytes CUDA errors
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

# Critical: Enables memory saving so the 3B model fits in the P100's 16GB VRAM
model.gradient_checkpointing_enable()

config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from datasets import load_dataset
import urllib.request

# Download your data
urllib.request.urlretrieve("https://raw.githubusercontent.com/Youmei295/DeAIze-model-source-code/main/dataset.json", "dataset.json")
dataset = load_dataset("json", data_files="dataset.json", split="train")

def formatting_func(examples):
    texts = []
    for inp, tar in zip(examples["input"], examples["target"]):
        messages = [
            {"role": "system", "content": "You are a professional Vietnamese editor. Rewrite the text to be natural, human-sounding, and concise. Remove robotic AI-isms and repetitive transitions."},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": tar},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 1, # Ultra low to fit in P100 16GB VRAM
    gradient_accumulation_steps = 8, # Compensate for small batch size
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 5e-5,
    fp16 = True, # Use standard FP16 (P100 supported)
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_torch", # Standard optimizer, no bitsandbytes needed
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048, # Reduced sequence length to save memory
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

trainer.train()

In [ ]:
# Save and push the LoRA adapters to the Hugging Face Hub
model.push_to_hub("Youmei295/deAIze", token=hf_token)
tokenizer.push_to_hub("Youmei295/deAIze", token=hf_token)

In [ ]:
import datetime
import json
from github import Github

# 1. Run Evaluation on a Fixed Example
fixed_example = "CỤ ÔNG 61 TUỔI “HỦY D.IỆT” CẢ GIẢI ĐẤU… CHỈ VÌ KHÔNG BIẾT LUẬT ĐƯỢC PHÉP NGỦ =))\nNăm đó, khi cụ ông Cliff Young xuất hiện tại vạch xuất phát của giải marathon siêu dài, ai cũng tưởng cụ tới… xem cho vui.\nGiữa dàn vận động viên chuyên nghiệp mặc đồ thể thao xịn sò, cụ lại diện nguyên bộ bảo hộ lao động cùng đôi ủng nhà nông cũ kỹ. Nhìn kiểu gì cũng không giống người đi thi.\nNhiều người còn cười nhạo khi thấy dáng chạy “lết lết” kỳ lạ của cụ.\nNhưng rồi cuộc đời thích chơi plot twist.\nTrong khi các chân chạy hàng đầu áp dụng chiến thuật khoa học: chạy 18 tiếng rồi ngủ 6 tiếng để hồi sức, thì cụ Cliff lại hồn nhiên nghĩ:\n“Đua là phải chạy liên tục chứ?”\nVà thế là cụ… cắm đầu chạy xuyên đêm.\nNgày này qua ngày khác.\nKhông nghỉ.\nKhông ngủ.\nHóa ra nhiều năm lùa cừu ngoài trang trại đã vô tình biến cụ thành “quái vật sức bền” thực thụ. Dáng chạy chậm chạp kia lại cực kỳ tiết kiệm năng lượng, giúp cụ đều đặn tiến về phía trước khi tất cả đối thủ bắt đầu kiệt sức.\nKết quả, Cliff Young cán đích đầu tiên sau 5 ngày 15 giờ 4 phút, bỏ xa người về nhì hơn 10 tiếng đồng hồ và tạo nên một trong những cú sốc lớn nhất lịch sử marathon.\nNhưng điều khiến người ta nể phục nhất lại đến sau chiến thắng.\nKhi nhận 10.000 đô tiền thưởng, cụ ngơ ngác hỏi:\n“Mọi người không được chia tiền à?”\nRồi cụ đem chia sạch cho các đối thủ phía sau vì nghĩ ai cũng đã cố gắng như mình.\nMột huyền thoại đúng nghĩa:\nKhông chiến thuật.\nKhông hào nhoáng.\nChỉ có sự bền bỉ, chân thành và một trái tim quá đẹp."
messages = [
    {"role": "system", "content": "You are a professional Vietnamese editor. Rewrite the text to be natural, human-sounding, and concise. Remove robotic AI-isms and repetitive transitions."},
    {"role": "user", "content": fixed_example}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9, repetition_penalty=1.05)

generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# 2. Log Result to GitHub
try:
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    if github_token:
        g = Github(github_token)
        repo = g.get_repo("Youmei295/DeAIze-model-source-code")
        
        log_data = {
            "timestamp": datetime.datetime.now().isoformat(),
            "input": fixed_example,
            "output": generated_text
        }
        
        file_name = f"logs/eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        content = json.dumps(log_data, indent=2, ensure_ascii=False)
        
        repo.create_file(
            path=file_name,
            message="Automated evaluation log after retraining",
            content=content,
            branch="main"
        )
        print(f"Successfully logged evaluation to {file_name}")
    else:
        print("GITHUB_TOKEN secret not found or empty, skipping logging.")
except Exception as e:
    print(f"Error accessing Kaggle secrets or GitHub: {e}")
